## 1. Import Required Libraries and Set Paths

This cell imports essential Python libraries for file and path handling (os, shutil, Path), running external commands (subprocess), and parsing JSON data (json). It also locates the ffmpeg and ffprobe executables, which you installed separately on your system. These tools are used to analyze and convert video files. The code uses shutil.which() to automatically find the installed executables in your system’s PATH.

Note: Before running this code, you must have installed FFmpeg on your system:

Windows: Download the FFmpeg full build from ffmpeg.org
, extract it, and add the bin folder to your system PATH.

Linux: Use your package manager, e.g., sudo apt install ffmpeg.

Mac: Use Homebrew: brew install ffmpeg.

The variables FFMPEG and FFPROBE now point to the installed executables and will be used in later cells for checking and converting video files.

In [26]:
import os
import subprocess
import json
import shutil
from pathlib import Path

# Paths to ffmpeg and ffprobe (dynamically locate or fallback to provided paths)
FFMPEG = shutil.which('ffmpeg') 
FFPROBE = shutil.which('ffprobe') 

## 2. Define Festival Specifications

This cell defines the video and audio format specifications required by the Narbonne Film Festival. It includes container format, video and audio codecs, frame rate, resolution, and bitrate specifications.

In [27]:
# Define festival format specifications
FESTIVAL_SPEC = {
    'container': 'mp4',
    'container_aliases': ['mp4', 'mov', 'm4a', 'm4v'],  # Acceptable container aliases
    'video_codec': 'h264',
    'audio_codec': 'aac',
    'frame_rate': 25.0,
    'aspect_ratio': '16:9',
    'resolution': (640, 360),
    'video_bitrate_min': 2000000,  # 2 Mb/s
    'video_bitrate_max': 5000000,  # 5 Mb/s
    'audio_bitrate_max': 256000,   # 256 kb/s
    'audio_channels': 2  # Stereo
}

## 3. Get Video Information Using ffprobe

This cell defines a function that runs ffprobe to extract detailed video and audio stream information from a file. The result is returned in JSON format, allowing for easy parsing of video attributes.

In [28]:
def get_video_info(file_path):
    """Extract video and audio stream information using ffprobe."""
    cmd = [
        FFPROBE, "-v", "error", "-show_entries", "format=bit_rate,format_name:stream=index,codec_name,codec_type,width,height,r_frame_rate,display_aspect_ratio,bit_rate,channels",
        "-of", "json", str(file_path)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return json.loads(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error analyzing {file_path}: {e}")
        return None

## 4. Parse Bitrate for Proper Conversion

This function converts a bitrate string into an integer value. It handles empty or invalid inputs gracefully by returning 0, making it robust for parsing metadata where bitrates might be missing or malformed. It is used to handle video/audio bitrate fields, which are often provided as strings in the output of ffprobe.

In [29]:
def parse_bitrate(bitrate_str):
    """Convert bitrate string to integer (bits/s)."""
    if not bitrate_str:
        return 0
    try:
        return int(bitrate_str)
    except ValueError:
        return 0

## 5. Print File Specifications

This function prints a formatted summary of a video file's specifications, such as container type, video/audio codecs, frame rate, resolution, bitrates, and more. It relies on the previously defined get_video_info to fetch metadata and parse_bitrate to handle bitrate conversion, making it a utility for displaying media details in a human-readable format.

In [30]:
def print_file_specs(file_path, label="Specifications"):
    """Print detailed specifications of a video file."""
    info = get_video_info(file_path)
    if not info or 'streams' not in info or 'format' not in info:
        print(f"{label} for {file_path}: Unable to retrieve specifications")
        return
    print(f"\n{label} for {file_path}:")
    print("-" * 50)
    container_list = info['format']['format_name'].split(',')
    # If file has .mp4 extension and mp4 is in aliases, use mp4 as container
    if Path(file_path).suffix.lower() == '.mp4' and 'mp4' in container_list:
        container = 'mp4'
    else:
        container = container_list[0]  # Take first alias otherwise
    print(f"Container: {container}")
    video_stream = None
    audio_stream = None
    for stream in info['streams']:
        if stream['codec_type'] == 'video':
            video_stream = stream
        elif stream['codec_type'] == 'audio':
            audio_stream = stream
    if video_stream:
        print(f"Video Codec: {video_stream['codec_name']}")
        try:
            num, denom = map(int, video_stream['r_frame_rate'].split('/'))
            frame_rate = num / denom if denom != 0 else 0
            print(f"Frame Rate: {frame_rate:.2f} FPS")
        except (KeyError, ValueError):
            print("Frame Rate: Unknown")
        width, height = int(video_stream.get('width', 0)), int(video_stream.get('height', 0))
        print(f"Resolution: {width}x{height}")
        display_aspect_ratio = video_stream.get('display_aspect_ratio', f"{width}:{height}")
        print(f"Aspect Ratio: {display_aspect_ratio}")
        video_bitrate = parse_bitrate(video_stream.get('bit_rate', info['format'].get('bit_rate', '0')))
        print(f"Video Bitrate: {video_bitrate/1000:.0f} kb/s")
    else:
        print("Video Stream: Not found")
    if audio_stream:
        print(f"Audio Codec: {audio_stream['codec_name']}")
        audio_bitrate = parse_bitrate(audio_stream.get('bit_rate', '0'))
        print(f"Audio Bitrate: {audio_bitrate/1000:.0f} kb/s")
        channels = int(audio_stream.get('channels', 0))
        print(f"Audio Channels: {channels}")
    else:
        print("Audio Stream: Not found")
    print("-" * 50)

## 6. Check Video Format Against Festival Specifications

This function, `check_format(file_path)`, analyzes a video file's metadata (obtained via `get_video_info`) to ensure it meets predefined festival specifications stored in `FESTIVAL_SPEC`, checking elements like container format (with special handling for .mp4 files), video codec, frame rate (with error and warning thresholds for deviations), resolution, aspect ratio, and bitrate, as well as audio codec, bitrate, and channels; it identifies video and audio streams from the metadata, collects critical errors for non-compliance and warnings for potential issues like playback smoothness, and returns a dictionary summarizing the file path, errors, and warnings for easy reporting or further processing.

In [31]:
def check_format(file_path):
    """Check if the video file complies with festival specifications."""
    info = get_video_info(file_path)
    if not info or 'streams' not in info or 'format' not in info:
        return {'file': file_path, 'errors': ['Unable to analyze file'], 'warnings': []}
    errors = []
    warnings = []
    video_stream = None
    audio_stream = None
    # Check container
    container_list = info['format']['format_name'].split(',')
    # If file has .mp4 extension and mp4 is in aliases, treat as mp4
    if Path(file_path).suffix.lower() == '.mp4' and 'mp4' in container_list:
        container = 'mp4'
    else:
        container = container_list[0]
    if container != FESTIVAL_SPEC['container']:
        errors.append(f"Container is {container}, expected {FESTIVAL_SPEC['container']}")
    # Find video and audio streams
    for stream in info['streams']:
        if stream['codec_type'] == 'video':
            video_stream = stream
        elif stream['codec_type'] == 'audio':
            audio_stream = stream
    # Check video stream
    if video_stream:
        if video_stream['codec_name'] != FESTIVAL_SPEC['video_codec']:
            errors.append(f"Video codec is {video_stream['codec_name']}, expected {FESTIVAL_SPEC['video_codec']}")
        # Check frame rate
        try:
            num, denom = map(int, video_stream['r_frame_rate'].split('/'))
            frame_rate = num / denom if denom != 0 else 0
            if abs(frame_rate - FESTIVAL_SPEC['frame_rate']) > 0.01:
                errors.append(f"Frame rate is {frame_rate:.2f}, expected {FESTIVAL_SPEC['frame_rate']}")
            if abs(frame_rate - FESTIVAL_SPEC['frame_rate']) > 0.5:
                warnings.append(f"Significant frame rate change from {frame_rate:.2f} to {FESTIVAL_SPEC['frame_rate']} FPS may affect playback smoothness")
        except (KeyError, ValueError):
            errors.append("Unable to determine frame rate")
        # Check resolution
        width, height = int(video_stream.get('width', 0)), int(video_stream.get('height', 0))
        if (width, height) != FESTIVAL_SPEC['resolution']:
            errors.append(f"Resolution is {width}x{height}, expected {FESTIVAL_SPEC['resolution'][0]}x{FESTIVAL_SPEC['resolution'][1]}")
        # Check aspect ratio
        display_aspect_ratio = video_stream.get('display_aspect_ratio', f"{width}:{height}")
        if display_aspect_ratio != FESTIVAL_SPEC['aspect_ratio']:
            errors.append(f"Aspect ratio is {display_aspect_ratio}, expected {FESTIVAL_SPEC['aspect_ratio']}")
        # Check video bitrate
        video_bitrate = parse_bitrate(video_stream.get('bit_rate', info['format'].get('bit_rate', '0')))
        if not (FESTIVAL_SPEC['video_bitrate_min'] <= video_bitrate <= FESTIVAL_SPEC['video_bitrate_max']):
            errors.append(f"Video bitrate is {video_bitrate/1000:.0f} kb/s, expected {FESTIVAL_SPEC['video_bitrate_min']/1000:.0f}-{FESTIVAL_SPEC['video_bitrate_max']/1000:.0f} kb/s")
    else:
        errors.append("No video stream found")
    # Check audio stream
    if audio_stream:
        if audio_stream['codec_name'] != FESTIVAL_SPEC['audio_codec']:
            errors.append(f"Audio codec is {audio_stream['codec_name']}, expected {FESTIVAL_SPEC['audio_codec']}")
        audio_bitrate = parse_bitrate(audio_stream.get('bit_rate', '0'))
        if audio_bitrate > FESTIVAL_SPEC['audio_bitrate_max']:
            errors.append(f"Audio bitrate is {audio_bitrate/1000:.0f} kb/s, expected up to {FESTIVAL_SPEC['audio_bitrate_max']/1000:.0f} kb/s")
        channels = int(audio_stream.get('channels', 0))
        if channels != FESTIVAL_SPEC['audio_channels']:
            errors.append(f"Audio channels is {channels}, expected {FESTIVAL_SPEC['audio_channels']} (stereo)")
    else:
        errors.append("No audio stream found")
    return {'file': file_path, 'errors': errors, 'warnings': warnings}

## 7. Convert Video to Meet Specifications

The `convert_video` function employs several video processing and transcoding techniques to ensure compliance with festival specifications while optimizing efficiency and quality. First, it uses **metadata analysis** via `get_video_info` to extract detailed stream information, allowing precise identification of video and audio streams without re-encoding unnecessary parts. **Error categorization** is applied by filtering the `check_result` errors into video-specific (e.g., codec, frame rate, resolution, aspect ratio, bitrate) and audio-specific (e.g., codec, bitrate, channels) issues, enabling **selective transcoding**: compliant streams are copied directly using `-c:v copy` or `-c:a copy` to preserve original quality and reduce processing time, while non-compliant ones are re-encoded. For video transcoding, it leverages **H.264 encoding** with the libx264 codec, incorporating **pixel format conversion** to yuv420p for broad compatibility, and **video bitrate control** through average bitrate (`-b:v 4800k`), maximum bitrate (`-maxrate 5000k`), and buffer size (`-bufsize 10000k`) to manage data rate and prevent buffering issues during playback, although the comment mentions two-pass encoding for better VBR control, the implemented command uses single-pass constant bitrate-like settings. **Frame rate adjustment** is handled with `-r` to force the specified FPS, potentially resampling frames if needed. **Resolution scaling** uses `-s` to resize the video to the target dimensions, which may involve interpolation techniques like bilinear or lanczos (default in FFmpeg). **Aspect ratio correction** applies `-aspect` to set the display aspect ratio (DAR), which FFmpeg uses to adjust pixel aspect ratio (PAR) without necessarily stretching pixels. For audio, it applies **AAC encoding** with a fixed bitrate (`-b:a 192k`) for compression efficiency and **channel remapping** via `-ac` to enforce stereo if required. The output is forced into an **MP4 container** with `-f mp4` and branded as 'mp42' for ISO Base Media File Format compliance, ensuring device compatibility. Finally, the function executes the dynamically built FFmpeg command via `subprocess.run` with error capturing, providing a robust, platform-agnostic way to run the external process, and returns the output file or an error message, promoting automation in batch processing workflows. 

In [32]:
def convert_video(file_path, check_result):
    """Convert video to festival specifications and save with '_formatOK.mp4' suffix."""
    output_file = str(Path(file_path).with_stem(Path(file_path).stem + '_formatOK').with_suffix('.mp4'))
    errors = check_result['errors']
    
    info = get_video_info(file_path)
    if not info:
        return None, "Unable to get video info"
    
    video_stream = next((s for s in info['streams'] if s['codec_type'] == 'video'), None)
    audio_stream = next((s for s in info['streams'] if s['codec_type'] == 'audio'), None)
    
    if not video_stream or not audio_stream:
        return None, "Missing video or audio stream"
    
    # Determine if video needs transcoding
    video_errors = [e for e in errors if e.startswith(('Video codec', 'Frame rate', 'Resolution', 'Aspect ratio', 'Video bitrate'))]
    video_needs_transcode = bool(video_errors) or any(e.startswith('No video') for e in errors)
    
    # Determine if audio needs transcoding
    audio_errors = [e for e in errors if e.startswith(('Audio codec', 'Audio bitrate', 'Audio channels'))]
    audio_needs_transcode = bool(audio_errors) or any(e.startswith('No audio') for e in errors)
    
    cmd = [FFMPEG, '-y', '-i', str(file_path)]
    
    # Video
    if not video_needs_transcode:
        cmd += ['-c:v', 'copy']
    else:
        # Use two-pass encoding for better bitrate control
        cmd += ['-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-b:v', '4800k', '-maxrate', '5000k', '-bufsize', '10000k']
        
        # Frame rate
        if any('Frame rate' in e for e in errors):
            cmd += ['-r', str(FESTIVAL_SPEC['frame_rate'])] 
        
        # Resolution
        if any('Resolution' in e for e in errors):
            cmd += ['-s', f"{FESTIVAL_SPEC['resolution'][0]}x{FESTIVAL_SPEC['resolution'][1]}"]
        
        # Aspect ratio
        if any('Aspect ratio' in e for e in errors):
            cmd += ['-aspect', FESTIVAL_SPEC['aspect_ratio']]
        
    # Audio
    if not audio_needs_transcode:
        cmd += ['-c:a', 'copy']
    else:
        cmd += ['-c:a', 'aac', '-b:a', '192k']  # Use a lower audio bitrate if needed
        
        # Channels
        if any('Audio channels' in e for e in errors):
            cmd += ['-ac', str(FESTIVAL_SPEC['audio_channels'])]
    
    # Container
    cmd += ['-f', 'mp4', '-brand', 'mp42', output_file]
    
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"Converted {file_path} to {output_file}")
        return output_file, None
    except subprocess.CalledProcessError as e:
        error_msg = f"ffmpeg error: {e.stderr}"
        print(f"Error converting {file_path}: {e}")
        print(error_msg)
        return None, error_msg


## 8. Generate Report and Convert Non-Compliant Files

This cell generates a detailed report that logs which films comply with the festival's format and which require conversion. It provides a summary of the results and performs conversion for non-compliant files

In [33]:
def generate_report(film_files, report_file='format_report.txt'):
    """Generate a report of non-compliant films and convert them."""
    passed_count = 0
    failed_count = 0
    total_files = len(film_files)
    summary = []

    with open(report_file, 'w') as f:
        f.write("Narbonne Online Film Festival - Format Compliance Report\n")
        f.write("=" * 50 + "\n\n")

        # Loop through the film files to check and generate the summary
        for file in film_files:
            # Check format and store the result
            result = check_format(file)
            
            # Determine if the file passed or failed
            if result['errors']:
                failed_count += 1
                status = "Failed"
            else:
                passed_count += 1
                status = "Passed"
            
            # Add the file name and its status to the summary list
            summary.append(f"{file}: {status}")
        
        # Write the summary at the top
        f.write(f"Total files: {total_files}\n")
        f.write(f"Files Passed: {passed_count}\n")
        f.write(f"Files Failed: {failed_count}\n\n")

        # Print each file's name and its pass/fail status in the summary
        f.write("File Status Summary:\n")
        for item in summary:
            f.write(f" - {item}\n")
        
        f.write("\n" + "=" * 50 + "\n\n")

        # Loop through the film files for detailed analysis
        for file in film_files:
            # Print original specs
            print_file_specs(file, "Original Specifications")
            
            # Check format and write to report
            result = check_format(file)
            f.write(f"File: {result['file']}\n")
            
            if result['errors']:
                f.write("Issues found:\n")
                for error in result['errors']:
                    f.write(f" - {error}\n")
                f.write(f"Conversion Required: {file}\n")
            else:
                f.write("File complies with festival specifications\n")
            
            # Convert file if necessary
            if result['errors']:  # Only convert if there are errors
                f.write("Converting file...\n")
                output_file, error_msg = convert_video(file, result)
                if output_file:
                    f.write(f"Converted file saved as {output_file}\n")
                    # Print specs of converted file
                    print_file_specs(output_file, "Converted Specifications")
                else:
                    f.write(f"Conversion failed: {error_msg}\n")
            
            f.write("\n" + "-" * 50 + "\n")

        # Update the summary at the end of the report
        f.write("\nSummary:\n")
        f.write("=" * 50 + "\n")
        f.write(f"Total files: {total_files}\n") 
        f.write(f"Conversion Summary: {failed_count} files converted\n")

## 9. Running the Code

This cell executes the report generation process. It defines a list of files, runs the generate_report function, and generates the compliance report. It also prints a confirmation once the report is created

In [34]:
# List of provided film files
film_files = [
    'Cosmos_War_of_the_Planets.mp4',
    'Last_man_on_earth_1964.mov',
    'The_Gun_and_the_Pulpit.avi',
    'The_Hill_Gang_Rides_Again.mp4',
    'Voyage_to_the_Planet_of_Prehistoric_Women.mp4'
]

# Generate report and convert non-compliant files
report_file = 'format_report.txt'
generate_report(film_files, report_file=report_file)
print(f"Report generated as '{report_file}'")



Original Specifications for Cosmos_War_of_the_Planets.mp4:
--------------------------------------------------
Container: mp4
Video Codec: h264
Frame Rate: 29.97 FPS
Resolution: 628x354
Aspect Ratio: 314:177
Video Bitrate: 2989 kb/s
Audio Codec: aac
Audio Bitrate: 317 kb/s
Audio Channels: 2
--------------------------------------------------
Converted Cosmos_War_of_the_Planets.mp4 to Cosmos_War_of_the_Planets_formatOK.mp4

Converted Specifications for Cosmos_War_of_the_Planets_formatOK.mp4:
--------------------------------------------------
Container: mp4
Video Codec: h264
Frame Rate: 25.00 FPS
Resolution: 640x360
Aspect Ratio: 16:9
Video Bitrate: 4722 kb/s
Audio Codec: aac
Audio Bitrate: 193 kb/s
Audio Channels: 2
--------------------------------------------------

Original Specifications for Last_man_on_earth_1964.mov:
--------------------------------------------------
Container: mov
Video Codec: prores
Frame Rate: 23.98 FPS
Resolution: 640x360
Aspect Ratio: 16:9
Video Bitrate: 9285 k

## 10. Check Converted File Compliance

This cell defines the check_converted_format function, which validates if the converted video files meet the festival's format specifications. It checks properties like container format, video and audio codecs, frame rate, resolution, bitrate, and audio channels. The function returns any errors or warnings if the files do not comply with the required standards. This is useful for ensuring that all converted files are in the correct format before final submission.

In [19]:
def check_converted_format(file_path):
    """Check if the converted video file complies with festival specifications."""
    info = get_video_info(file_path)
    if not info or 'streams' not in info or 'format' not in info:
        return {'file': file_path, 'errors': ['Unable to analyze file'], 'warnings': []}
    errors = []
    warnings = []
    video_stream = None
    audio_stream = None
    
    # Check container
    container_list = info['format']['format_name'].split(',')
    if Path(file_path).suffix.lower() == '.mp4' and 'mp4' in container_list:
        container = 'mp4'
    else:
        container = container_list[0]
    if container != FESTIVAL_SPEC['container']:
        errors.append(f"Container is {container}, expected {FESTIVAL_SPEC['container']}")
    
    # Find video and audio streams
    for stream in info['streams']:
        if stream['codec_type'] == 'video':
            video_stream = stream
        elif stream['codec_type'] == 'audio':
            audio_stream = stream
    
    # Check video stream
    if video_stream:
        if video_stream['codec_name'] != FESTIVAL_SPEC['video_codec']:
            errors.append(f"Video codec is {video_stream['codec_name']}, expected {FESTIVAL_SPEC['video_codec']}")
        
        try:
            num, denom = map(int, video_stream['r_frame_rate'].split('/'))
            frame_rate = num / denom if denom != 0 else 0
            if abs(frame_rate - FESTIVAL_SPEC['frame_rate']) > 0.01:
                errors.append(f"Frame rate is {frame_rate:.2f}, expected {FESTIVAL_SPEC['frame_rate']}")
        except (KeyError, ValueError):
            errors.append("Unable to determine frame rate")
        
        width, height = int(video_stream.get('width', 0)), int(video_stream.get('height', 0))
        if (width, height) != FESTIVAL_SPEC['resolution']:
            errors.append(f"Resolution is {width}x{height}, expected {FESTIVAL_SPEC['resolution'][0]}x{FESTIVAL_SPEC['resolution'][1]}")
        
        display_aspect_ratio = video_stream.get('display_aspect_ratio', f"{width}:{height}")
        if display_aspect_ratio != FESTIVAL_SPEC['aspect_ratio']:
            errors.append(f"Aspect ratio is {display_aspect_ratio}, expected {FESTIVAL_SPEC['aspect_ratio']}")
        
        video_bitrate = parse_bitrate(video_stream.get('bit_rate', info['format'].get('bit_rate', '0')))
        if not (FESTIVAL_SPEC['video_bitrate_min'] <= video_bitrate <= FESTIVAL_SPEC['video_bitrate_max']):
            errors.append(f"Video bitrate is {video_bitrate/1000:.0f} kb/s, expected {FESTIVAL_SPEC['video_bitrate_min']/1000:.0f}-{FESTIVAL_SPEC['video_bitrate_max']/1000:.0f} kb/s")
    else:
        errors.append("No video stream found")
    
    # Check audio stream
    if audio_stream:
        if audio_stream['codec_name'] != FESTIVAL_SPEC['audio_codec']:
            errors.append(f"Audio codec is {audio_stream['codec_name']}, expected {FESTIVAL_SPEC['audio_codec']}")
        audio_bitrate = parse_bitrate(audio_stream.get('bit_rate', '0'))
        if audio_bitrate > FESTIVAL_SPEC['audio_bitrate_max']:
            errors.append(f"Audio bitrate is {audio_bitrate/1000:.0f} kb/s, expected up to {FESTIVAL_SPEC['audio_bitrate_max']/1000:.0f} kb/s")
        channels = int(audio_stream.get('channels', 0))
        if channels != FESTIVAL_SPEC['audio_channels']:
            errors.append(f"Audio channels is {channels}, expected {FESTIVAL_SPEC['audio_channels']} (stereo)")
    else:
        errors.append("No audio stream found")
    
    return {'file': file_path, 'errors': errors, 'warnings': warnings}

# Example usage:
converted_files = [
    'Cosmos_War_of_the_Planets_formatOK.mp4',
    'Last_man_on_earth_1964_formatOK.mp4',
    'The_Gun_and_the_Pulpit_formatOK.mp4',
    'The_Hill_Gang_Rides_Again_formatOK.mp4',
    'Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4'
]

# Check the converted files
for file in converted_files:
    result = check_converted_format(file)
    if result['errors']:
        print(f"{file} failed the compliance check with the following errors:")
        for error in result['errors']:
            print(f"  - {error}")
    else:
        print(f"{file} passed the compliance check.")


Cosmos_War_of_the_Planets_formatOK.mp4 passed the compliance check.
Last_man_on_earth_1964_formatOK.mp4 passed the compliance check.
The_Gun_and_the_Pulpit_formatOK.mp4 passed the compliance check.
The_Hill_Gang_Rides_Again_formatOK.mp4 passed the compliance check.
Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4 passed the compliance check.
